# Chat Completions API Fundamentals

This notebook explains how the OpenAI chat completions API works and demonstrates how to build simple interactions using structured messages.

## Learning Goals

- Understand the `chat.completions.create()` call
- Learn the role of `system`, `user`, and `assistant` messages
- Experiment with parameters like `temperature`
- Inspect responses and token usuage
- Build a multi-turn conversation

In [ ]:
# Install dependencies
%pip install openai python-dotenv       

In [ ]:
# Imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# Load environment variables
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("No API key found in environment variables")

client = OpenAI(api_key=api_key)
print("API key loaded successfully")

## What is the Chat Completions API?

The Chat Completions API generates responses based on a sequence of messages.

A typical API call looks like this:

- `model` : which model to use
- `messages` : the conversation history
- `temperature` : controls randomness/creativity

The API reads the messages in order and produces the next assistant response.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Explain what an API is in simple terms."}
    ],
    temperature=0.3,
)

reply = response.choices[0].message.content
print(reply)

## Anatomy of the Request

This request has three important parts:

### 1. Model
Specifies which model should generate the response.

### 2. Messages
A list of conversation turns. Each message has:
- a `role`
- a `content`

### 3. Temperature
Controls randomness:
- low temperature = more focused and consistent
- high temperature = more creative and variable

## Message Roles

### System
Defines the assistant's behavior, style, or constraints.

### User
Represents the human's input.

### Assistant
Represents previous model responses in a multi-turn conversation.

In [ ]:
# Compare different system prompts
prompts = [
    "You are a helpful assistant.",
    "You are a senior Python teacher who explain things simply.",
    "You are a strict technical reviewer who answers concisely."
]

question = "What is a function in Python?"

for system_prompt in prompts:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}   
        ],
        temperature=0.3,
    )
    print("=" * 80)
    print("SYSTEM PROMPT: ", system_prompt)
    print()
    print(response.choices[0].message.content)
    print()

In [ ]:
# Compare temperature values
question = "Give me a short explanation of machine learning."

for temp in [0.0, 0.3, 0.7, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question}
        ],
        temperature=temp,
    )
    print("=" * 80)
    print(f"Temperature: {temp}")
    print()
    print(response.choices[0].message.content)
    print()
    

In [ ]:
# Inspect the full response object
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.3,
)

# Print the response object
response



## Understanding the response

Importsnt fields to inspect:

-`choices[0].message.content` : the assistant's reply
- `usage` : token counts
- `model` : the model used

In [ ]:
# Token usage
if response.usage:
    print(f"Prompt tokens: {response.usage.prompt_tokens}")
    print(f"Completion tokens: {response.usage.completion_tokens}")
    print(f"Total tokens: {response.usage.total_tokens}")

# Model used


## Why Token Usuage Matters

Token usage affects:
- cost
- latency
- context window usage

In production systems, tracking tokens is essential.

## Multi-turn Conversation

The model does not automatically remember past messages unless you send them again in the `messages` list.
Conversation memory is created by including previous user assistant messages.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "I am learning AI engineering."},
    {"role": "assistant", "content": "That's great. AI engineering combines software engineering and machine kearning systems."},
    {"role": "user", "content": "What should I learn first?"}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.3
)

print(response.choices[0].message.content)

In [ ]:
# Why chat history matters
question = "What should I learn first?"
response_no_history = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": question}
    ],
    temperature=0.3,
)

print("Without history:\n")
print(response_no_history.choices[0].message.content)

In [ ]:
# Structured prompting example
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": """
            Explain retrieval-augmented generation in this format:

            1. Definition
            2. Why it is useful
            3. Simple example
            4. One limitation
            """
        
        }
    ],
    temperature=0.3,
)

display(Markdown(response.choices[0].message.content))

## Practical Notes

- The model only knows the messages you provide in the request.
- Better prompts usually produce better outputs.
- Long conversations use more tokens
- For production systems, prompts should be clear, testable, and cost-aware.

## Key Takeways

- The Chat Completions API generates responses from a sequence of messages.
- `system` messages guide behavior.
- `user` messages contain tasks or questions.
- `assistant` messages preserve prior responses in multi-turn chat.
- `temperature` affects randomness
- `usage` helps monitor cost and performance